# BI Agent Tests

Use this notebook to validate the BI agent end to end: question → SQL result → Plotly chart + narrative.

The BI agent consumes a `QueryExecutionResult` produced by the Data Engineer agent,
so both agents are exercised here.

Before running the notebook, make sure:
- `data/askdata.db` exists
- `GOOGLE_API_KEY` is available in the environment used by the notebook kernel

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path("..").resolve()
dotenv_path = project_root / ".env"

if not project_root.exists():
    raise FileNotFoundError(
        "Could not find the AskData project root from the current notebook working directory."
    )

if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

load_dotenv(dotenv_path=dotenv_path, override=True)

print(f"Project root: {project_root}")
print(f"Loaded environment from: {dotenv_path}")
print(f"GOOGLE_API_KEY loaded: {bool(os.getenv('GOOGLE_API_KEY'))}")

Project root: /Users/zac_burns/Documents/Personal/AskData
Loaded environment from: /Users/zac_burns/Documents/Personal/AskData/.env
GOOGLE_API_KEY loaded: True


In [2]:
from askdata.agents import AgentConfig, DataEngineerAgent
from askdata.agents.bi_agent import BIAgent
from askdata.storage import SQLiteDatabase

In [3]:
database = SQLiteDatabase(project_root / "data" / "askdata.db")
database.ensure_exists()

print(f"Database path: {database.db_path}")
print("Available tables:", database.list_tables())

Database path: /Users/zac_burns/Documents/Personal/AskData/data/askdata.db
Available tables: ['customers', 'geolocation', 'order_items', 'orders', 'payments', 'products', 'sellers', 'walmart_trustpilot_reviews']


In [4]:
de_agent = DataEngineerAgent(config=AgentConfig(thinking_level="low"))
bi_agent = BIAgent(config=AgentConfig())

print("Data Engineer agent:", de_agent)
print("BI agent:", bi_agent)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Data Engineer agent: <askdata.agents.data_engineer.DataEngineerAgent object at 0x11ec91570>
BI agent: <askdata.agents.bi_agent.BIAgent object at 0x11ec913c0>


## Helper

Runs the full pipeline: question → SQL → DataFrame → chart + narrative.

In [5]:
def analyze(question: str) -> None:
    sep = "─" * 60

    # Step 1: Data Engineer generates SQL and executes it
    de_result = de_agent.run(question)
    print(f"{'📝 Generated SQL':^60}")
    print(sep)
    print(de_result.sql)
    print()
    print(f"{'📊 Raw data'} — {len(de_result.dataframe):,} rows")
    print(sep)
    display(de_result.dataframe.head(10))
    print()

    # Step 2: BI Agent builds chart + narrative
    bi_result = bi_agent.run(de_result)
    print(f"{'💬 Narrative':^60}")
    print(sep)
    print(bi_result.narrative)
    print()

    for fig in bi_result.charts:
        fig.show()

## Example 1 — Order volume by status

A simple grouping question that should produce a bar chart and a short insight about delivery vs. cancellation rates.

In [6]:
analyze("How many orders do we have by status? Order by the most frequent.")

                      📝 Generated SQL                       
────────────────────────────────────────────────────────────
SELECT order_status, COUNT(order_id) AS order_count FROM orders GROUP BY order_status ORDER BY order_count DESC

📊 Raw data — 8 rows
────────────────────────────────────────────────────────────


,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


KeyboardInterrupt: 

## Example 2 — Revenue trend over time

A time-series question. Expect a line chart and a narrative about growth or seasonality.

In [ ]:
analyze(
    "What is the total revenue per month over the entire dataset? Show months in order."
)